# Sanity Check

Verifies that all three dynamical systems and both complexity measures behave correctly **without requiring any pre-generated HDF5 files**.

Run this notebook first after cloning the repository.

**Contents**
1. Logistic map — bifurcation diagram
2. Hénon map — phase portrait of the attractor
3. Rössler system — 3-D attractor
4. PE and ETC on test signals
5. MLE spot-checks at known parameter values

In [ ]:
import sys
from pathlib import Path

# Allow running from any working directory
ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib
matplotlib.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

FIG_DIR = ROOT / "figures" / "sanity"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", dpi=300, bbox_inches="tight")
    print(f"Saved → figures/sanity/{name}.png/.pdf")

print("Imports OK. ROOT =", ROOT)

## 1. Logistic map — bifurcation diagram

Expected: period-doubling cascade leading to chaos around `a ≈ 3.57`, with visible periodic windows (notably the period-3 window near `a ≈ 3.83`).

In [ ]:
from src.maps.logistic import simulate

a_values = np.linspace(3.4, 4.0, 800)
L_bif, transient = 300, 500

a_plot, x_plot = [], []
for a in a_values:
    s = simulate(a, L_bif, transient)
    a_plot.extend([a] * L_bif)
    x_plot.extend(s.tolist())

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(a_plot, x_plot, ',k', alpha=0.08, markersize=0.5)
ax.set(xlabel="Parameter  a", ylabel="x",
       title="Logistic Map — Bifurcation Diagram")
plt.tight_layout()
savefig(fig, "logistic_bifurcation")
plt.show()

## 2. Hénon map — phase portrait

Expected: the classic banana-shaped strange attractor at `a=1.4, b=0.3`.

In [ ]:
from src.maps.henon import simulate as henon_simulate

# Return full (x, y) trajectory for plotting — call _trajectory directly
from src.maps.henon import _trajectory

rng = np.random.default_rng(42)
x0, y0 = rng.uniform(-0.5, 0.5, 2)
s = _trajectory(1.4, 0.3, 80_000, 1000, x0, y0)

# _trajectory only returns x; simulate the y series manually for the portrait
def henon_xy(a, b, L, transient, seed=42):
    rng = np.random.default_rng(seed)
    x0, y0 = rng.uniform(-0.5, 0.5), rng.uniform(-0.5, 0.5)
    x, y = x0, y0
    for _ in range(transient):
        x, y = 1.0 - a*x*x + y, b*x
    xs, ys = np.zeros(L), np.zeros(L)
    for i in range(L):
        x, y = 1.0 - a*x*x + y, b*x
        xs[i], ys[i] = x, y
    return xs, ys

xs, ys = henon_xy(1.4, 0.3, 80_000, 1000)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(xs, ys, ',k', alpha=0.15, markersize=0.4)
ax.set(xlabel="x", ylabel="y", title="Hénon Attractor  (a=1.4, b=0.3)")
plt.tight_layout()
savefig(fig, "henon_attractor")
plt.show()

## 3. Rössler system — 3-D attractor

Expected: a folded-band strange attractor. The x-component (used in all analyses) is shown in red along the bottom.

In [ ]:
from scipy.integrate import solve_ivp

def rossler(t, s, a, b, c):
    x, y, z = s
    return [-y-z, x+a*y, b+z*(x-c)]

a, b, c = 0.2, 0.2, 5.7
t_end = 3000.0
sol = solve_ivp(rossler, [0, t_end], [1.0, 0.0, 0.0],
                args=(a, b, c), method='RK45',
                t_eval=np.linspace(500, t_end, 50_000),
                rtol=1e-9, atol=1e-9)
x, y, z = sol.y

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot(x, y, z, 'steelblue', lw=0.3, alpha=0.7)
ax.set(xlabel="x", ylabel="y", zlabel="z",
       title="Rössler Attractor  (a=b=0.2, c=5.7)")
plt.tight_layout()
savefig(fig, "rossler_attractor_3d")
plt.show()

## 4. PE and ETC on known signals

Expected behaviour:
- **Constant signal** → PE = 0, ETC ≈ 0
- **White noise** → PE ≈ 1, ETC ≈ high (not 1)
- **Sine wave** → PE and ETC intermediate, consistent with periodic order
- **Logistic chaos (a=4)** → PE and ETC high

In [ ]:
from src.complexity import pe_method, etc_method
from src.maps.logistic import simulate as logistic_sim

rng = np.random.default_rng(0)
L_test = 5000

signals = {
    "Constant":       np.ones(L_test) * 0.5,
    "White noise":    rng.random(L_test),
    "Sine wave":      0.5 + 0.4 * np.sin(2*np.pi*np.arange(L_test)/50),
    "Logistic (a=4)": logistic_sim(4.0, L_test, 1000),
}

D, bins = 3, 2
print(f"{'Signal':<20}  {'PE (D='+str(D)+')':<12}  {'ETC (bins='+str(bins)+')'}")
print("-" * 50)
for name, sig in signals.items():
    pe  = pe_method(sig, D)
    etc = etc_method(sig, bins)
    print(f"{name:<20}  {pe:<12.4f}  {etc:.4f}")

## 5. MLE spot-checks

Expected:
- Logistic `a=3.5`: negative or near-zero (period-4 orbit)
- Logistic `a=4.0`: positive, close to `ln(2) ≈ 0.693`
- Hénon `a=1.4, b=0.3`: positive (~0.42 is the literature value)
- Rössler `a=b=0.2, c=5.7`: positive (~0.07 is the literature value)

In [ ]:
from src.maps.logistic import lyapunov_mle as logistic_mle
from src.maps.henon    import lyapunov_mle as henon_mle
from src.maps.rossler  import lyapunov_mle as rossler_mle

L_mle = 200_000

checks = [
    ("Logistic a=3.5",          logistic_mle(3.5, L_mle, 1000),       "< 0 (period-4)"),
    ("Logistic a=4.0",          logistic_mle(4.0, L_mle, 1000),       "≈ 0.693 (ln 2)"),
    ("Hénon   a=1.4, b=0.3",    henon_mle(1.4, 0.3, L_mle, 1000),    "≈ 0.42"),
    ("Rössler a=b=0.2, c=5.7",  rossler_mle(0.2, 0.2, 5.7, 50_000),  "≈ 0.07"),
]

print(f"{'System':<28}  {'MLE':>10}  {'Expected'}")
print("-" * 60)
for label, mle, expected in checks:
    print(f"{label:<28}  {mle:>10.4f}  {expected}")